## Creating a Knowledge graph from text

In this notebook the framework [LightRAG](https://lightrag.github.io]) will be used to generate a knowledge graph from a text containing a little story.

LightRAG makes several calls to a LLM in order to get entities, relations and answers to questions. In this example a model with 70 billions parameters will be used. Since a large model like this can't be run locally, a model running in Groq platform will be used. Additionally, LightRAG needs to use an embedding model to get embeddings of text chunks and also of the queries. We will use Ollama to run the embedding model locally.

## Install Ollama

Follow the following steps to download Ollama and the embedding model.

- Download Ollama form https://ollama.com/download and install it. Check that Ollama is running after installation (`ollama ps`).
- From the system shell, download the embedding model: `ollama pull nomic-embed-text`

## Install LightRAG and additional Python libraries

In order to avoid conflicts between the needed libraries and existing libraries in your machine, it will be better to use a Python environment. In this case will be used a Conda environment.

````
conda create -n lightrag126 python=3.12
conda activate lightrag126
curl -L -O https://github.com/HKUDS/LightRAG/archive/refs/tags/v1.2.6.zip
unzip v1.2.6.zip
cd LightRAG-1.2.6
pip install -e .
pip install networkx
pip install matplotlib
pip install pyvis
conda install conda-forge::textract # to use textract to read txt, pdf, ...

## File to generate the knowledge graph

Copy to the current dir the file containing the story (`threeLittlePigs.txt`) or other text file from your preference.

## LLM cloud provider

In this notebook we will be using the [Groq Cloud](https://console.groq.com) to run the LLM. This service provides a free tear allowing to use different models throw an Open AI compatible API.

Create a free account and then generate an API key. After getting the API key, we need to place it in the code in a cell bellow.

You can browse the list of available models in Groq Cloud and make experiments with some of them. Check the usage limits of each model (number of API calls per second and number of tokens processed by day).

In [ ]:
import os
import glob
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt
import webbrowser
import numpy as np
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.llm.ollama import ollama_embed
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status
import textract

## Don't forget to insert your GROQ API KEY ...

In [ ]:
WORKING_DIR = "./threeLittlePigs"
doc_path = "./threeLittlePigs.txt" # path to the document to be indexed

# Set your API key for the GROQ API
os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

def remove_graph():
    files = glob.glob(WORKING_DIR + '/*')
    for f in files:
        os.remove(f)

Generate a graph representation in html format from the graphml file and display it in the default browser.

In [ ]:
def show_graph():
    """
    Loads a graph from a GraphML file, visualizes it using NetworkX and Matplotlib,
    and displays the visualization in the default browser.
    """
    file_path = WORKING_DIR + '/' + 'graph_chunk_entity_relation.graphml'

    # Load the GraphML file
    G = nx.read_graphml(file_path)

    # Create a Pyvis network
    net = Network(notebook=True, cdn_resources='remote')

    # Convert NetworkX graph to Pyvis network
    net.from_nx(G, show_edge_weights=True)

    # Save and display the network
    net.show('knowledge_graph.html')
    net.show_buttons(filter_=['physics'])

    filepath = os.getcwd()
    file_uri = 'file:///' + filepath + '/' 'knowledge_graph.html'
    webbrowser.open(file_uri)
    

Create a PNG image file with the representation of the graph, generated from the graphml file (xml format)

In [ ]:
def save_graph():
    """
    Loads a graph from a graphml file, visualizes it using NetworkX and Matplotlib,
    and saves the visualization to a PNG file.
    """
    figure = plt.figure(figsize=(10, 10))

    file_path = WORKING_DIR + '/' + 'graph_chunk_entity_relation.graphml'

    if not os.path.exists(file_path):
        print("Graph file not found:" + file_path)
        return

    G = nx.read_graphml(file_path)

    #pos=nx.spring_layout(G)
    #pos=nx.spectral_layout(G)
    #pos=nx.circular_layout(G)
    pos=nx.shell_layout(G)

    # Draw the graph
    nx.draw(G, pos, with_labels=True, node_size=700, node_color='skyblue', font_size=10, font_color='black', font_weight='bold', edge_color='gray')

    labels = nx.get_edge_attributes(G,'weight')
    # labels = nx.get_edge_attributes(G,'description')
    nx.draw_networkx_edge_labels(G,pos,edge_labels=labels)

    plt.show()
    
    plt.tight_layout()
    plt.savefig(WORKING_DIR + '/' + 'graph.png',dpi=200)

Functions need to create the graph rag

In [ ]:
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    return await openai_complete_if_cache(
        #"mixtral-8x7b-32768",
        "llama-3.3-70b-versatile",
        #"deepseek-r1-distill-llama-70b",
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
        **kwargs,
    )


async def embedding_func(texts: list[str]) -> np.ndarray:
    return await ollama_embed(
    #return await openai_embed(
        texts,
        embed_model="nomic-embed-text",
        host="http://localhost:11434",
    )


async def get_embedding_dim():
    test_text = ["This is a test sentence."]
    embedding = await embedding_func(test_text)
    embedding = np.asarray(embedding, dtype=np.float32)
    embedding_dim = embedding.shape[1]
    return embedding_dim

async def initialize_rag():
    embedding_dimension = await get_embedding_dim()
    print(f"Detected embedding dimension: {embedding_dimension}")

    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_model_func,
        embedding_func=EmbeddingFunc(
            embedding_dim=embedding_dimension,
            max_token_size=8192,
            func=embedding_func,
        ),
    )

    await rag.initialize_storages()
    await initialize_pipeline_status()

    return rag

Initialize graph rag, insert a document, export the graph to a Excel spreadsheet, and make a query. Try different types of queries. Consult the [LightRAG](https://lightrag.github.io]) repo.

In [ ]:
# Remove a previous generated graph
remove_graph()

# Initialize LightRAG
rag = await initialize_rag()

# Insert data from file into the graph
with open(doc_path, "r", encoding="utf-8") as f:
    await rag.ainsert(f.read())
#await rag.ainsert(textract.process(doc_path).decode("utf-8"))

# Export data in Excel sheet
await rag.aexport_data("graph_data.xlsx", file_format="excel")
    
# Make a query to the graph
result = await rag.aquery("What are the top themes in this story?", param=QueryParam(mode="global"))
print(result)

# Show graph in the default browser
#show_graph()

Display the graph.

In [ ]:
show_graph()

In [ ]:
save_graph()

In [ ]:
# merge entities and relations
await rag.amerge_entities(
    source_entities=["Third Little Pig", "The Third Little Pig"],
    target_entity="Third Little Pig"
)

In [ ]:
await rag.amerge_entities(
    source_entities=["Second Little Pig", "The Second Little Pig"],
    target_entity="Second Little Pig"
)

Some times, the knowledge graph includes duplicated entities with different names, such as:

"The Straw House" and "Straw House".

The function `merge_entities_with_common_prefix(prefix)` merge such entities, where its names differ only in the prefix. In the previous example, the two entities will be merged in the entity "Straw House". 

In [ ]:
file_path = WORKING_DIR + '/' + 'graph_chunk_entity_relation.graphml'

def get_entities_different_by_prefix(prefix):
    def get_entities(file):
        G = nx.read_graphml(file_path)
        return list(G.nodes)
    
    entities = get_entities(file_path)
    entities_to_merge = []
    for entity in entities:
        if entity.startswith(prefix):
            entity_without_prefix = entity[len(prefix):].strip()
            if entity_without_prefix in entities:
                entities_to_merge.append((entity, entity_without_prefix))

    return entities_to_merge

async def merge_entities_with_common_prefix(prefix):
    entities_to_merge = get_entities_different_by_prefix(prefix)
    for source_entity, target_entity in entities_to_merge:
        await rag.amerge_entities(
            source_entities=[source_entity],
            target_entity=target_entity
        )

await merge_entities_with_common_prefix("The")
show_graph()

## Insert a custom knowledge graph

It is possible to define the graph manually (entities and relations).

In this example, we defined the entities, the relations between entities, and the source text chunks.

In [ ]:
WORKING_DIR = "./custom_kg"

if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

# Initialize a new LightRAG object
rag2 = await initialize_rag()

custom_kg = {
    "entities": [
        {
            "entity_name": "First Little Pig",
            "entity_type": "person",
            "description": "The first little pig is lazy and builds his house out of straw, which is easily blown down by the wolf.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Second Little Pig",
            "entity_type": "person",
            "description": "The second little pig works a little harder than the first, but still builds his house out of sticks, which is also blown down by the wolf.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Third Little Pig",
            "entity_type": "person",
            "description": "The third little pig is hardworking and builds his house out of bricks, which withstands the wolf's attempts to blow it down.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Wolf",
            "entity_type": "person",
            "description": "The wolf is the antagonist who tries to eat the three little pigs, but ultimately fails and is boiled and eaten by them.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Stick House",
            "entity_type": "geo",
            "description": "The stick house is the home built by the second little pig, which is also blown down by the wolf.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Straw House",
            "entity_type": "geo",
            "description": "The straw house is the home built by the first little pig, which is easily blown down by the wolf.",
            "source_id": "Source1",
        },
        {
            "entity_name": "Brick House",
            "entity_type": "geo",
            "description": "The brick house is the sturdy home built by the third little pig, where the three little pigs take refuge from the wolf.",
            "source_id": "Source1",
        }
    ],
    "relationships": [
        {
            "src_id": "First Little Pig",
            "tgt_id": "Wolf",
            "description": "The wolf tries to eat the first little pig, but the pig escapes and runs to the second little pig's house.",
            "keywords": "predator-prey, survival",
            "weight": 16.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Second Little Pig",
            "tgt_id": "Wolf",
            "description": "The wolf also tries to eat the second little pig, but the pig escapes and runs to the third little pig's house.<SEP>The wolf tries to eat the second little pig, but the pig escapes and runs to the third little pig's house.",
            "keywords": "employment, affiliation",
            "weight": 16.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Third Little Pig",
            "tgt_id": "Wolf",
            "description": "The third little pig outsmarts the wolf and ultimately kills him by boiling him in a pot of water.",
            "keywords": "outsmarting, survival",
            "weight": 18.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Straw House",
            "tgt_id": "First Little Pig",
            "description": "The first little pig builds his house out of straw, which is easily blown down by the wolf.",
            "keywords": "construction, vulnerability",
            "weight": 4.0, 
            "source_id": "Source1",
        },
        {
            "src_id": "Stick House",
            "tgt_id": "Second Little Pig",
            "description": "The second little pig builds his house out of sticks, which is also blown down by the wolf.",
            "keywords": "construction, vulnerability",
            "weight": 4.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Brick House",
            "tgt_id": "Third Little Pig",
            "description": "The third little pig builds his house out of bricks, which withstands the wolf's attempts to blow it down.",
            "keywords": "construction, strength",
            "weight": 6.0,
            "source_id": "Source1",
        },
        {
            "src_id": "First Little Pig",
            "tgt_id": "Straw House",
            "description": "The first little pig builds his house out of straw, which is easily blown down by the wolf.",
            "keywords": "construction, vulnerability",
            "weight": 4.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Second Little Pig",
            "tgt_id": "Stick House",
            "description": "The second little pig builds his house out of sticks, which is also blown down by the wolf.",
            "keywords": "construction, vulnerability",
            "weight": 4.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Wolf",
            "tgt_id": "Straw House",
            "description": "The wolf tries to eat the first little pig, but the pig escapes and runs to the second little pig's house.",
            "keywords": "predator-prey, survival",
            "weight": 16.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Wolf",
            "tgt_id": "Stick House",
            "description": "The wolf also tries to eat the second little pig, but the pig escapes and runs to the third little pig's house.<SEP>The wolf tries to eat the second little pig, but the pig escapes and runs to the third little pig's house.",
            "keywords": "predator-prey, survival",
            "weight": 16.0,
            "source_id": "Source1",
        },
        {
            "src_id": "Wolf",
            "tgt_id": "Brick House",
            "description": "The wolf tries to blow down the brick house, but fails, and ultimately meets his demise when he tries to enter through the chimney.",
            "keywords": "foiled plan, demise",
            "weight": 16.0,
            "source_id": "Source1",
        }
    ],
"chunks": [
        {"content": "The Three Little Pigs.","source_id": "Source1","source_chunk_index": 0,},
        {"content": "Once upon a time there was an old mother pig who had three little pigs and not enough food to feed them.","source_id": "Source1","source_chunk_index": 1,},
        {"content": "So when they were old enough, she sent them out into the world to seek their fortunes.","source_id": "Source1","source_chunk_index": 2,},
        {"content": "The first little pig was very lazy.","source_id": "Source1","source_chunk_index": 3,},
        {"content": "He didn't want to work at all and he built his house out of straw.","source_id": "Source1","source_chunk_index": 4,},
        {"content": "The second little pig worked a little bit harder but he was somewhat lazy too and he built his house out of sticks.","source_id": "Source1","source_chunk_index": 5,},
        {"content": "Then, they sang and danced and played together the rest of the day.","source_id": "Source1","source_chunk_index": 6,},
        {"content": "The third little pig worked hard all day and built his house with bricks.","source_id": "Source1","source_chunk_index": 7,},
        {"content": "It was a sturdy house complete with a fine fireplace and chimney.","source_id": "Source1","source_chunk_index": 8,},
        {"content": "It looked like it could withstand the strongest winds.","source_id": "Source1","source_chunk_index": 9,},
        {"content": "The next day, a wolf happened to pass by the lane where the three little pigs lived; and he saw the straw house, and he smelled the pig inside.","source_id": "Source1","source_chunk_index": 10,},
        {"content": "He thought the pig would make a mighty fine meal and his mouth began to water.","source_id": "Source1","source_chunk_index": 11,},
        {"content": "So he knocked on the door and said:  Little pig! Little pig! Let me in! Let me in!.","source_id": "Source1","source_chunk_index": 12,},
        {"content": "But the little pig saw the wolf's big paws through the keyhole, so he answered back:  No! No! No!  Not by the hairs on my chinny chin chin!.","source_id": "Source1","source_chunk_index": 13,},
        {"content": "Three Little Pigs, the straw house Then the wolf showed his teeth and said:  Then I'll huff  and I'll puff  and I'll blow your house down.","source_id": "Source1","source_chunk_index": 14,},
        {"content": "So he huffed and he puffed and he blew the house down! The wolf opened his jaws very wide and bit down as hard as he could, but the first little pig escaped and ran away to hide with the second little pig.","source_id": "Source1","source_chunk_index": 15,},
        {"content": "The wolf continued down the lane and he passed by the second house made of sticks; and he saw the house, and he smelled the pigs inside, and his mouth began to water as he thought about the fine dinner they would make.","source_id": "Source1","source_chunk_index": 16,},
        {"content": "So he knocked on the door and said:    Little pigs! Little pigs!   Let me in! Let me in!.","source_id": "Source1","source_chunk_index": 17,},
        {"content": "But the little pigs saw the wolf's pointy ears through the keyhole, so they answered back:  No! No! No! Not by the hairs on our chinny chin chin!.","source_id": "Source1","source_chunk_index": 18,},
        {"content": "So the wolf showed his teeth and said:  Then I'll huff  and I'll puff  and I'll blow your house down!.","source_id": "Source1","source_chunk_index": 19,},
        {"content": "So he huffed and he puffed and he blew the house down! The wolf was greedy and he tried to catch both pigs at once, but he was too greedy and got neither! His big jaws clamped down on nothing but air and the two little pigs scrambled away as fast as their little hooves would carry them.","source_id": "Source1","source_chunk_index": 20,},
        {"content": "The wolf chased them down the lane and he almost caught them.","source_id": "Source1","source_chunk_index": 21,},
        {"content": "But they made it to the brick house and slammed the door closed before the wolf could catch them.","source_id": "Source1","source_chunk_index": 22,},
        {"content": "The three little pigs they were very frightened, they knew the wolf wanted to eat them.","source_id": "Source1","source_chunk_index": 23,},
        {"content": "And that was very, very true.","source_id": "Source1","source_chunk_index": 24,},
        {"content": "The wolf hadn't eaten all day and he had worked up a large appetite chasing the pigs around and now he could smell all three of them inside and he knew that the three little pigs would make a lovely feast.","source_id": "Source1","source_chunk_index": 25,},
        {"content": "Three Little Pigs at the Brick House.","source_id": "Source1","source_chunk_index": 26,},
        {"content": "So the wolf knocked on the door and said:  Little pigs! Little pigs! Let me in! Let me in!.","source_id": "Source1","source_chunk_index": 27,},
        {"content": "But the little pigs saw the wolf's narrow eyes through the keyhole, so they answered back:  No! No! No!  Not by the hairs on our chinny chin chin!.","source_id": "Source1","source_chunk_index": 28,},
        {"content": "So the wolf showed his teeth and said:  Then I'll huff  and I'll puff  and I'll blow your house down.","source_id": "Source1","source_chunk_index": 29,},
        {"content": "Well! he huffed and he puffed.","source_id": "Source1","source_chunk_index": 30,},
        {"content": "He puffed and he huffed.","source_id": "Source1","source_chunk_index": 31,},
        {"content": "And he huffed, huffed, and he puffed, puffed; but he could not blow the house down.","source_id": "Source1","source_chunk_index": 32,},
        {"content": "At last, he was so out of breath that he couldn't huff and he couldn't puff anymore.","source_id": "Source1","source_chunk_index": 33,},
        {"content": "So he stopped to rest and thought a bit.","source_id": "Source1","source_chunk_index": 34,},
        {"content": "But this was too much.","source_id": "Source1","source_chunk_index": 35,},
        {"content": "The wolf danced about with rage and swore he would come down the chimney and eat up the little pig for his supper.","source_id": "Source1","source_chunk_index": 36,},
        {"content": "But while he was climbing on to the roof the little pig made up a blazing fire and put on a big pot full of water to boil.","source_id": "Source1","source_chunk_index": 37,},
        {"content": "Then, just as the wolf was coming down the chimney, the little piggy pulled off the lid, and plop! in fell the wolf into the scalding water.","source_id": "Source1","source_chunk_index": 38,},
        {"content": "So the little piggy put on the cover again, boiled the wolf up, and the three little pigs ate him for supper.","source_id": "Source1","source_chunk_index": 39,},
        {"content": "Three Little Pigs, the wolf falls into the pot.","source_id": "Source1","source_chunk_index": 40,},
        {"content": "None","source_id": "UNKNOWN", "source_chunk_index": 0,},
    ],    
    
}

await rag2.ainsert_custom_kg(custom_kg)

save_graph()

result = await rag2.aquery("Who is the owner of the brick house?", param=QueryParam(mode="global"))
print(result)

Another query over the custom graph.

In [ ]:
result = await rag2.aquery("Who fall on the pot?", param=QueryParam(mode="local"))
print(result)